# Funciones, sus gradientes, espacio de exploración y mínimos globales

Función de Rosenbrock:

$$ f(x,y) = (1-x)^2 + 100(y-x^2)^2 $$

Gradiente:

$$ \nabla f = [2(1-x) - 400x(y-x^2)]\hat{i} + 200(y-x^2)\hat{j} $$

Tiene mínimo global $0$ en $(1,1)$.

Función _Three-Hump Camel_:

$$ f(x,y) = 2x^2 - 1.05x^4 + \tfrac{x^6}{6} + xy + y^2 $$

Gradiente: 

$$ \nabla f = [4x - 4.20x^3 + x^5 + y]\hat{i} + (1 + 2y)\hat{j} $$

Mínimo global $0$ en $(0, 0)$, función definida en $x_i \in [-5, 5]$.

Función _Booth_:

$$ f(x,y) = (x+2y-7)^2 + (2x + y - 5)^2 $$

Gradiente: 

$$ \nabla f = [2(x + 2y - 7) + 4(2x + y - 5)]\hat{i} + [4(x + 2y - 7) + 2(2x + y - 5)]\hat{j} $$

$$ \nabla f = [10x + 8y - 34]\hat{i} + [8x + 10y - 38]\hat{j} $$

Mínimo global $0$ en $(1,3)$, función suele ser evaluada en $x_i \in [-10, 10]$.

Función Styblinski-Tang:

$$ f(x) = \tfrac{1}{2} \sum_{i=1}^d x_i^4 - 16x_i^2 + 5x_i $$

$$ \nabla f = \tfrac{1}{2} \sum_{i=1}^d 4x_i^3 - 32x_i + 5 $$

Yo consideré $ d=2 $ por simplicidad.

Mínimo global $-39.16599d$ en $(-2.903534, \dots, -2.903534)$.

Suele ser evaluada en el hípercubo $x_i \in [-5, 5]$

Vamos a explorar todas en el segmento $x,y \in [-5, 5]$

Importamos las bibliotecas de siempre

In [20]:
import torch
import sklearn
import matplotlib.pyplot as plt
import numpy as np 

In [21]:
DIMS = 2 # Vamos a usar 2 dimensiones para todas las funciones
torch.manual_seed(21) # Para resultados reproducibles

# Codificando las funciones (criterios (pérdida))

In [57]:
def rosenbrock(x, y):
    return (1-x)**2 + 100 * (y - x**2)**2

def three_hump_camel(x, y):
    return 2*x**2 - 1.05 * x**4 + (1 / 6)*x**6 + x*y + y**2

def booth(x, y):
    return (x + 2*y - 7)**2 + (2*x + y - 5)**2

def styblinski_tang(*args):
    xis = torch.stack(list(args))

    return 0.5*(xis**4 - 16*xis**2 + 5*xis).sum()

Ya que queremos comparar los diferentes métodos de descenso haremos que sea más o menos justo y le daremos el mismo punto de pártida a todos los métodos:

In [31]:
puntos_iniciales = np.random.normal(size=(DIMS,))
puntos_iniciales

array([ 0.86564012, -1.06307494])

## Descenso de gradiente de una capa

In [61]:
def train_gd(punto_partida, criterio, lr, n_epocas=100):
    punto_actual = punto_partida.clone().detach().requires_grad_(True)
    historial = [punto_actual.detach()]
    for epoca in range(n_epocas):

        if punto_actual.grad is not None:
            punto_actual.grad.zero_()

        perdida = criterio(*punto_actual)
        perdida.backward()
        #optimizador.paso() no podemos utilizar el paso :c 
        # lo que vamos a hacer es hacer el descenso de gradiente manual con los valores de backward
        with torch.no_grad():
            punto_actual -= punto_actual.grad * lr
        

        historial.append(punto_actual.clone().detach())

        if epoca % 5 == 0 or epoca < 5:
            print(f"Pérdida {perdida.item()} en época {epoca + 1} en el punto ({punto_actual[0]}, {punto_actual[1]})")

    return punto_actual, historial

Ahora probaremos las funciones con lr, que parece funcionar más o menos bien 

In [62]:
optimo, historial = train_gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), rosenbrock, 1e-3)

Pérdida 328.500244140625 en época 1 en el punto (0.2383517026901245, -0.7005933523178101)
Pérdida 57.946319580078125 en época 2 en el punto (0.16766351461410522, -0.5491123795509338)
Pérdida 34.011470794677734 en época 3 en el punto (0.1306164562702179, -0.43366768956184387)
Pérdida 21.07143211364746 en época 4 en el punto (0.10880620777606964, -0.34352201223373413)
Pérdida 13.422357559204102 en época 5 en el punto (0.09512241184711456, -0.2724498510360718)
Pérdida 8.742923736572266 en época 6 en el punto (0.08622145652770996, -0.2161502242088318)
Pérdida 1.6733808517456055 en época 11 en el punto (0.07172347605228424, -0.06688357144594193)
Pérdida 0.947350800037384 en época 16 en el punto (0.07403089851140976, -0.018410375341773033)
Pérdida 0.8585570454597473 en época 21 en el punto (0.08077547699213028, -0.0020316713489592075)
Pérdida 0.8350130915641785 en época 26 en el punto (0.08890305459499359, 0.004158176481723785)
Pérdida 0.8183507919311523 en época 31 en el punto (0.0974232181

In [50]:
optimo, historial = train_gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), three_hump_camel, 1e-3)

Pérdida 1.1891040802001953 en época 1
Pérdida 1.187490463256836 en época 2
Pérdida 1.185882329940796 en época 3
Pérdida 1.1842796802520752 en época 4
Pérdida 1.1826822757720947 en época 5
Pérdida 1.1810901165008545 en época 6
Pérdida 1.1732075214385986 en época 11
Pérdida 1.165452003479004 en época 16
Pérdida 1.1578197479248047 en época 21
Pérdida 1.1503069400787354 en época 26
Pérdida 1.1429094076156616 en época 31
Pérdida 1.1356230974197388 en época 36
Pérdida 1.1284444332122803 en época 41
Pérdida 1.1213696002960205 en época 46
Pérdida 1.1143949031829834 en época 51
Pérdida 1.1075162887573242 en época 56
Pérdida 1.1007306575775146 en época 61
Pérdida 1.09403395652771 en época 66
Pérdida 1.0874229669570923 en época 71
Pérdida 1.0808939933776855 en época 76
Pérdida 1.0744435787200928 en época 81
Pérdida 1.0680683851242065 en época 86
Pérdida 1.0617649555206299 en época 91
Pérdida 1.055530071258545 en época 96


In [52]:
optimo, historial = train_gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), booth, 1e-3)

Pérdida 87.00045776367188 en época 1
Pérdida 84.14112854003906 en época 2
Pérdida 81.38282012939453 en época 3
Pérdida 78.7219467163086 en época 4
Pérdida 76.155029296875 en época 5
Pérdida 73.67871856689453 en época 6
Pérdida 62.54731750488281 en época 11
Pérdida 53.242889404296875 en época 16
Pérdida 45.46240997314453 en época 21
Pérdida 38.95318603515625 en época 26
Pérdida 33.50448226928711 en época 31
Pérdida 28.9405517578125 en época 36
Pérdida 25.11484146118164 en época 41
Pérdida 21.905122756958008 en época 46
Pérdida 19.20947265625 en época 51
Pérdida 16.9428768157959 en época 56
Pérdida 15.034442901611328 en época 61
Pérdida 13.42504596710205 en época 66
Pérdida 12.065367698669434 en época 71
Pérdida 10.914280891418457 en época 76
Pérdida 9.937483787536621 en época 81
Pérdida 9.106358528137207 en época 86
Pérdida 8.397038459777832 en época 91
Pérdida 7.78961181640625 en época 96


In [58]:
optimo, historial = train_gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), styblinski_tang, 1e-3)

Pérdida -14.609931945800781 en época 1
Pérdida -15.005538940429688 en época 2
Pérdida -15.408830642700195 en época 3
Pérdida -15.819796562194824 en época 4
Pérdida -16.238422393798828 en época 5
Pérdida -16.66468048095703 en época 6
Pérdida -18.908761978149414 en época 11
Pérdida -21.333179473876953 en época 16
Pérdida -23.921249389648438 en época 21
Pérdida -26.64816665649414 en época 26
Pérdida -29.481250762939453 en época 31
Pérdida -32.38103485107422 en época 36
Pérdida -35.303165435791016 en época 41
Pérdida -38.20098876953125 en época 46
Pérdida -41.02851867675781 en época 51
Pérdida -43.743385314941406 en época 56
Pérdida -46.30933380126953 en época 61
Pérdida -48.6979866027832 en época 66
Pérdida -50.88958740234375 en época 71
Pérdida -52.87300109863281 en época 76
Pérdida -54.644874572753906 en época 81
Pérdida -56.20844650268555 en época 86
Pérdida -57.5721321105957 en época 91
Pérdida -58.74819564819336 en época 96


## Descenso de gradiente con momento de una capa

## Descenso ADAM de una capa